# Fine-tune one model

This notebook fine-tunes one model with one seed, evaluates the adapter with its own vLLM server, and prints the gate against the few-shot baseline of the same model. It has no logic of its own: every step calls `make`. To train another model or seed, change `MODEL` or `SEED` and run it again. `ROUND` is the round of the data (see `results/data_manifest.json`). A round after the first adds `-r` and the number to the names, as in `qwen3-4b-sft-r2-0`, and needs its own few-shot baseline, because its test_ood documents are other documents.

`FINAL` is False until the final run. Then the evaluation covers dev and dev_real only, and the baseline and the gate are skipped. Set `FINAL = True` once, for the run that makes the claim, to evaluate every split and print the gate.

On Kaggle, use GPU T4 x2 and turn Internet on. Training runs on both GPUs (`NUM_GPUS = 2`). The evaluation starts one vLLM server on each GPU (`SERVERS = 2`) and shares the documents between them. With `DRY_RUN = True` the notebook runs without a GPU. It skips the install and the clone, uses the checkout it is in, trains a tiny random model for two steps, evaluates against a fake server instead of vLLM, and writes to a temporary folder. A dry run uses a fake model, so its results say nothing about a real model. Never copy them into `results/`.

The cells run in order. Run them all in the same session, because the later cells need what the earlier ones installed.

In [ ]:
DRY_RUN = False
REPO_URL = "https://github.com/mgetech/open-llm-adaptation.git"
REPO_REF = "main"  # a tag or a commit
MODEL = "qwen3-4b"
SEED = 0
CHECK_MODEL = "qwen3-1.7b"
CHECK_STEPS = 3
NUM_GPUS = 2  # 1 for a single GPU
SERVERS = 2  # vLLM servers for the evaluation, one per GPU
ROUND = 2  # round of the data, listed in results/data_manifest.json
FINAL = False  # True only for the final run: then every split, the baseline and the gate run

GPUS = [] if DRY_RUN else [f"NUM_GPUS={NUM_GPUS}"]
ROUNDS = [f"ROUND={ROUND}"]
TAG = "" if ROUND == 1 else f"-r{ROUND}"  # what the round adds to the names
DRY = ["DRY_RUN=1"] if DRY_RUN else []
OFFLINE = ["OFFLINE=1"] if DRY_RUN else []
SMOKE = ["SMOKE=1"] if DRY_RUN else []
SPLITS = [] if FINAL else ["SPLITS=dev dev_real"]

In [ ]:
import os
import subprocess
import tempfile
from pathlib import Path


def run(*command, check=True, cwd=None):
    options = dict(stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    with subprocess.Popen(command, encoding="utf-8", errors="replace", **options) as process:
        for line in process.stdout:
            print(line, end="")
    if check and process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)


def make(*arguments, check=True):
    run("make", *arguments, check=check)


if DRY_RUN:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    WORK = Path(tempfile.mkdtemp(prefix="training-dry-run-"))
    ADAPTERS_DIR = WORK / "adapters"
    RUNS_DIR = WORK / "runs"
    OUT_DIR = WORK / "eval"
    RUN = f"smoke-{SEED}"
    SYSTEM = f"smoke-sft-{SEED}"
    BASELINE = []
else:
    REPO = Path("/kaggle/working/open-llm-adaptation")
    ADAPTERS_DIR = REPO / "adapters"
    RUNS_DIR = REPO / "results" / f"round{ROUND}" / "runs"
    OUT_DIR = REPO / "results" / f"round{ROUND}" / "eval"
    RUN = f"{MODEL}{TAG}-{SEED}"
    SYSTEM = f"{MODEL}-sft{TAG}-{SEED}"
    BASELINE = [f"BASELINE={MODEL}-fewshot{TAG}"]
print("repository:", REPO)
print("adapters:", ADAPTERS_DIR)
print("run manifests:", RUNS_DIR)
print("results:", OUT_DIR)

## Python 3.12

Kaggle's default Python may be older, so uv installs its own 3.12.

In [ ]:
if DRY_RUN:
    print("dry run: skipped")
else:
    run("pip", "install", "--quiet", "uv")
    run("uv", "python", "install", "3.12")

## Repository

The data is in the repository, so nothing else is downloaded. If the repository is already there, for example after a restart of this cell, it is updated and reused.

In [ ]:
if DRY_RUN:
    print("dry run: skipped")
else:
    if not REPO.exists():
        run("git", "clone", REPO_URL, str(REPO))
    run("git", "-C", str(REPO), "fetch", "--tags", "origin")
    run("git", "-C", str(REPO), "checkout", REPO_REF)
os.chdir(REPO)

## Project and vLLM

The project is installed from the lock file. vLLM is the version pinned in `configs/models.yaml`. It needs the same torch version as the training libraries, so one environment serves both. The training libraries are installed by the first `make train`.

In [ ]:
if DRY_RUN:
    print("dry run: skipped")
else:
    make("setup")
    make("install-vllm")

## Model checks

The licence, size and serving checks, and the pinned revisions against the Hub.

In [ ]:
make("models-check", *OFFLINE)

## Short check (optional, about 15 minutes)

Before the long run, this trains the small model for a few steps and evaluates the dev split with that adapter. It uses the same number of GPUs as the long run, so it shows early whether the GPUs, the 4-bit loading and the serving of an adapter work, and it installs the training libraries. Its names end in `-steps`, the number of steps, `-gpus` and the number of GPUs, so it never replaces a real run. Do not copy its files into `results/`.

In [ ]:
make(
    "train",
    f"MODEL={CHECK_MODEL}",
    f"SEED={SEED}",
    f"MAX_STEPS={CHECK_STEPS}",
    f"ADAPTERS_DIR={ADAPTERS_DIR.as_posix()}",
    f"RUNS_DIR={RUNS_DIR.as_posix()}",
    *GPUS,
    *ROUNDS,
    *SMOKE,
)

In [ ]:
make(
    "eval-sft",
    f"MODEL={CHECK_MODEL}",
    f"SEED={SEED}",
    f"MAX_STEPS={CHECK_STEPS}",
    *GPUS,
    *ROUNDS,
    f"SERVERS={SERVERS}",
    "SPLITS=dev",
    f"ADAPTERS_DIR={ADAPTERS_DIR.as_posix()}",
    f"OUT_DIR={OUT_DIR.as_posix()}",
    *DRY,
    *SMOKE,
)

## Training

QLoRA fine-tuning with the settings of `configs/train.yaml`. With `NUM_GPUS = 2`, each GPU holds the whole 4-bit model and trains on half of every batch, and the effective batch size stays 16. A checkpoint is written every 50 steps, and the loss on the dev_real documents is measured each time. Training stops when that loss has not improved for 3 checkpoints in a row, and the adapter is the checkpoint with the lowest loss. The dev_real documents are never trained on. If the session stops in the middle, run the cells again up to this one and add `RESUME=1` to the call below to continue from the last checkpoint.

In [ ]:
make(
    "train",
    f"MODEL={MODEL}",
    f"SEED={SEED}",
    f"ADAPTERS_DIR={ADAPTERS_DIR.as_posix()}",
    f"RUNS_DIR={RUNS_DIR.as_posix()}",
    *GPUS,
    *ROUNDS,
    *SMOKE,
)

## Evaluation

`make eval-sft` starts the servers with the base model and the adapter, one per GPU. Until the final run (`FINAL = False`) it runs dev and dev_real only. With `FINAL = True` it runs every split: dev, test_iid, test_ood, edge and gold, and gold_v2 and dev_real once those files exist. It uses the prompt the adapter was trained on, shares the documents of each split between the servers, and stops the servers at the end. With `SERVERS = 1` it uses one server on the first GPU.

In [ ]:
make(
    "eval-sft",
    f"MODEL={MODEL}",
    f"SEED={SEED}",
    *ROUNDS,
    f"SERVERS={SERVERS}",
    *SPLITS,
    f"ADAPTERS_DIR={ADAPTERS_DIR.as_posix()}",
    f"OUT_DIR={OUT_DIR.as_posix()}",
    *DRY,
    *SMOKE,
)

## Baseline for this round (final run, round 2 and later)

The fine-tuned model is compared with the few-shot baseline of the same model on test_ood. Round 1 has that baseline in the repository. Another round has other test_ood documents, so the baseline runs again on this round's dev and test_ood, with its own server. Dev is there to tune the confidence threshold, as for every system. This cell runs only with `FINAL = True`, and not for round 1.

In [ ]:
if FINAL and ROUND > 1:
    make(
        "baseline",
        f"MODEL={MODEL}",
        *ROUNDS,
        "SPLITS=dev test_ood",
        f"OUT_DIR={OUT_DIR.as_posix()}",
        *DRY,
    )

## Gate

The table compares the adapter with the few-shot baseline of the same model and round. A model that does not pass the gate is not shipped. It runs only with `FINAL = True`, because the gate needs the test splits.

In [ ]:
if FINAL:
    make("gate", f"SYSTEM={SYSTEM}", *BASELINE, f"OUT_DIR={OUT_DIR.as_posix()}", check=False)

## Results

Copy the run manifest from `results/round<round>/runs/` and the folder of the system from `results/round<round>/eval/tax-booking/` into the same places in the repository. For the final run of a round after the first, also copy the folder of the few-shot baseline (`<model>-fewshot-r<round>`). Do not copy files that have `-steps` in their name. The adapter stays out of the repository.

The first cell lists the files. The second one zips them for the Output tab: `results-<run>.zip` holds exactly the files to copy into the repository, and `adapter-<run>.zip` holds the adapter. Download the small one first.

In [ ]:
for path in sorted(RUNS_DIR.glob("*.json")):
    print(path)
for path in sorted(OUT_DIR.rglob("*.json")):
    print(path)
for path in sorted((ADAPTERS_DIR / "tax-booking" / RUN).glob("*")):
    print(path)

In [ ]:
if not DRY_RUN:
    WANTED = [
        f"results/round{ROUND}/runs/tax-booking-{RUN}.json",
        f"results/round{ROUND}/eval/tax-booking/{SYSTEM}",
    ]
    if FINAL and ROUND > 1:
        WANTED.append(f"results/round{ROUND}/eval/tax-booking/{MODEL}-fewshot{TAG}")
    RESULTS_ZIP = f"/kaggle/working/results-{RUN}.zip"
    ADAPTER_ZIP = f"/kaggle/working/adapter-{RUN}.zip"
    run("rm", "-f", RESULTS_ZIP, ADAPTER_ZIP)
    run("zip", "-r", RESULTS_ZIP, *WANTED, cwd=REPO)
    run("zip", "-r", ADAPTER_ZIP, f"adapters/tax-booking/{RUN}", cwd=REPO)
    run("ls", "-lh", RESULTS_ZIP, ADAPTER_ZIP)